In [112]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import re
from faker import Faker
import random
import numpy as np

In [3]:
def create_db_connection(host_name, user_name, user_password, db_name):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database = db_name
        )
        print("MySQL Database connection successful")
    except Error as err:
        print(f"Error: '{err}'")

    return connection


def execute_query(connection, query, values=None):
    cursor = connection.cursor()
    try:
        if values:
            if isinstance(values[0], (list, tuple)):  
                    cursor.executemany(query, values)
            else: 
                cursor.execute(query, values)
        else:
            cursor.execute(query)
        connection.commit()
        print("Query successful")
    except Error as err:
        print(f"Error: '{err}'")


## Fake data

In [67]:
file_path = "sendo_products.csv"  
df = pd.read_csv(file_path)
df_product = df[['productID', 'productName', 'productBrand', 'productPrice', 'productDescription']]

In [ ]:
# Init Faker
fake_E = Faker()
fake_V = Faker('vi_VN')

### Fake Product


In [ ]:
categories = [
    ('ASM01', 'Áo sơ mi', 'Nam'),
    ('AT01', 'Áo thun', 'Nam'),
    ('AK01', 'Áo khoác', 'Nam'),
    ('QJ01', 'Quần jean', 'Nam'),
    ('QS01', 'Quần short', 'Nam'),
    ('QD01', 'Quần đùi', 'Nam'),
    ('GTT01', 'Giày thể thao', 'Nam'),
    ('GT01', 'Giày tây', 'Nam'),
    ('GSD01', 'Giày sandal', 'Nam'),
    ('ASM02', 'Áo sơ mi', 'Nữ'),
    ('AT02', 'Áo thun', 'Nữ'),
    ('AK02', 'Áo khoác', 'Nữ'),
    ('QJ02', 'Quần jean', 'Nữ'),
    ('QS02', 'Quần short', 'Nữ'),
    ('QD02', 'Quần đùi', 'Nữ'),
    ('GTT02', 'Giày thể thao', 'Nữ'),
    ('GT02', 'Giày tây', 'Nữ'),
    ('GSD02', 'Giày sandal', 'Nữ'),
    ('ASM03', 'Áo sơ mi', 'Unisex'),
    ('AT03', 'Áo thun', 'Unisex'),
    ('AK03', 'Áo khoác', 'Unisex'),
    ('QJ03', 'Quần jean', 'Unisex'),
    ('QS03', 'Quần short', 'Unisex'),
    ('QD03', 'Quần đùi', 'Unisex'),
]

df_product.loc[:, 'categoryID'] = 'A'
for index, row in df_product.iterrows():
    product_name = row['productName'].lower()
    found = False
    
    for category_id, category_name, gender in categories:
        if category_name.lower() in product_name and gender.lower() in product_name:
                df_product.loc[index, 'categoryID'] = category_id
                found = True
                break
    

    if not found:
        for category_id, category_name, _ in categories:
            if category_name.lower() in product_name:
                df_product.loc[index, 'categoryID'] = re.sub(r'\d', '', category_id)  
                break

df_product.head()

In [ ]:
# Define sizes
sizes = ["S", "M", "L"]

# Duplicate rows with different sizes
expanded_rows = []
for _, row in df_product.iterrows():
    for size in sizes:
        new_row = row.copy()
        new_row["size"] = size
        new_row["productID"] = f"{row['productID']}{size}"  # Append size to productID
        expanded_rows.append(new_row)

df_product_new = pd.DataFrame(expanded_rows)

# replace Nan values with 'None'
df_product_new = df_product_new.fillna('None')
df_product_new.reset_index(drop=True, inplace=True)
df_product_new.head(10)



### Fake Customer


In [ ]:
# Num rows to fake
num_customers = 50

customers = [(f"KH{i+1:03d}", fake_V.name(), random.choice(['male', 'female', 'other']),
              fake_V.date_of_birth(minimum_age=18, maximum_age=60),
              fake_V.phone_number(), fake_E.address(), random.randint(1, 40)) for i in range(num_customers)]

df_customers = pd.DataFrame(customers, columns=["Customer_ID", "Name", "Gender", "Date_of_Birth", "Phone", "Address", 
                                                "Location_ID"])

# Add a column for Age
df_customers['Date_of_Birth'] = pd.to_datetime(df_customers['Date_of_Birth'], format='%Y-%m-%d')
df_customers['Age'] = 2025 - df_customers['Date_of_Birth'].dt.year


,Customer_ID,Name,Gender,Date_of_Birth,Phone,Address,Location_ID,Age
0,KH001,Quý ông Tùng Nguyễn,male,1972-07-29,+84-33-666453,"41975 Marc Oval Apt. 475\nJosephberg, OK 07974",37,53
1,KH002,Hoàng Quang Dương,male,1987-01-27,+84-53-949284,Unit 1272 Box 7038\nDPO AP 75103,34,38
2,KH003,Quý cô Duyên Mai,female,1983-03-01,+84-49-680 3067,"699 Marc Hills\nSarahfurt, CA 10135",29,42
3,KH004,Duyên Đặng,female,1999-10-07,04 0509 0739,"497 Nguyen Isle\nBowenton, LA 88226",38,26
4,KH005,Tùng Nguyễn,male,2004-01-12,+84-33-239 4861,"48334 Chad Land Apt. 343\nSouth Matthew, CA 38580",5,21
5,KH006,Nhật Bảo Đặng,female,1991-08-06,(04)970-3909,"9293 Jeffery Inlet Suite 046\nNorth Lindastad,...",30,34
6,KH007,Kim Vũ,other,1981-02-03,+84 39 4025983,"8170 Dustin Summit Apt. 990\nEast Victoria, MA...",4,44
7,KH008,Hoàng Mai,other,1977-03-07,(08) 0322 5004,"632 Allen Park\nCarmenchester, WV 34509",31,48
8,KH009,Hải Bùi,female,1964-04-13,01 2646805,233 Michael Light Suite 422\nPort Travischeste...,15,61
9,KH010,Bà Hồng Mai,male,1992-08-05,+84-82-538 7856,"5479 Robinson Springs Apt. 613\nWalkerchester,...",35,33


### Fake Location

In [ ]:
df_location = pd.read_csv('location.csv')

,district,province,Location_ID,nation
0,Ba Đình,Hà Nội,1,Vietnam
1,Hoàn Kiếm,Hà Nội,2,Vietnam
2,Tây Hồ,Hà Nội,3,Vietnam
3,Long Biên,Hà Nội,4,Vietnam
4,Cầu Giấy,Hà Nội,5,Vietnam


### Fake Inventory + ImportInventory

In [121]:
df_inventory = df_product_new['productID']
inventory = [(product_id, random.randint(1, 100)) for product_id in df_inventory]
import_inventory = [(product_id, random.randint(1, 100), fake_E.date_between(start_date='-2y')) for product_id in df_inventory]

### Fake OrderDetail

In [ ]:
num_order_details = 200
orderID = [f"DH{i+1:03d}" for i in range(101)]
# Dữ liệu cho OrderDetail (tránh trùng khóa chính)
order_detail_set = set()
order_details = []
while len(order_details) < num_order_details:
    order_id = random.choice(orderID)
    product_id = random.choice(df_product_new['productID'])
    if (order_id, product_id) not in order_detail_set:
        order_detail_set.add((order_id, product_id))
        order_details.append((order_id, product_id, random.randint(1, 5)))


order_details[:5]


[('DH033', '95123172L', 2),
 ('DH027', '116515329L', 4),
 ('DH003', '116541017S', 2),
 ('DH011', '119231216S', 1),
 ('DH013', '117849948S', 2)]

## Insert

### Insert OrderDetail

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = "INSERT INTO Order_Detail (OrderID, ProductID, Quantity) VALUES (%s, %s, %s)"
values = order_details
execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Error: '1146 (42S02): Table 'ecommerce_shop.orderdetail' doesn't exist'
Đã đóng kết nối MySQL


### Insert Inventory

In [120]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = "INSERT INTO Inventory (ProductID, InStock_Quantity) VALUES (%s, %s)"
values = inventory
execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Đã đóng kết nối MySQL


### Insert Import_Inventory


In [123]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = "INSERT INTO ImportInventory (ProductID, ImportQuantity, Date) VALUES (%s, %s, %s)"
values = import_inventory
execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Đã đóng kết nối MySQL


### Insert Location

In [103]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

for index, row in df_location.iterrows():
    
    insert_query = """
    INSERT INTO Location (ID, Nation, City, District) 
    VALUES (%s, %s, %s, %s)
    """   
    values = (
        row["Location_ID"], row["nation"], row["province"], row["district"]
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Đã đóng kết nối MySQL


### Insert Customer

In [89]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

for index, row in df_customers.iterrows():
    
    insert_query = """
    INSERT INTO Customer (ID, Name, Gender, DOB, phoneNumber, Address, locationID, Age) 
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """   
    values = (
        row["Customer_ID"], row["Name"], row["Gender"], row["Date_of_Birth"], row["Phone"], row["Address"], row["Location_ID"], row["Age"]
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Query successful
Đã đóng kết nối MySQL


### Insert Product

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")
customer_query = """
INSERT INTO Customer (ID, Name, Gender, DOB, phoneNumber, Address, locationID) VALUES (%s, %s, %s, %s, %s, %s)
"""
# import data into Product
for index, row in df_product_new.iterrows():
    
    insert_query = """
    INSERT INTO Product (ProductID, Name, Brand, Price, Description, Category, Size)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """   
    values = (
        row["productID"], row["productName"], row["productBrand"], row["productPrice"], row["productDescription"],
        row["categoryID"], row["size"]
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

### Insert Platform

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = """
INSERT INTO Platfrom (ID, Platform_Name, Platform_URL)
VALUES ('SENDO', 'Sendo', 'https://www.sendo.vn/'),
         ('TIKI', 'Tiki', 'https://tiki.vn/'),
         ('LAZADA', 'Lazada', 'https://www.lazada.vn/')  
"""
execute_query(connection, insert_query)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

### Insert Category


In [ ]:
# insert data into Category
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = """
INSERT INTO Category (ID, Name, Type)
    VALUES ('ASM01','Áo sơ mi', 'Nam'),
           ('AT01', 'Áo thun', 'Nam'),
           ('AK01', 'Áo khoác', 'Nam'),
           ('QJ01', 'Quần jean', 'Nam'),
           ('QS01', 'Quần short', 'Nam'),
           ('QD01', 'Quần đùi', 'Nam'),
           ('GTT01', 'Giày thể thao', 'Nam'),
           ('GT01', 'Giày tây', 'Nam'),
           ('GSD01', 'Giày sandal', 'Nam'),
           ('ASM02','Áo sơ mi', 'Nữ'),
           ('AT02', 'Áo thun', 'Nữ'),
           ('AK02', 'Áo khoác', 'Nữ'),
           ('QJ02', 'Quần jean', 'Nữ'),
           ('QS02', 'Quần short', 'Nữ'),
           ('QD02', 'Quần đùi', 'Nữ'),
           ('GTT02', 'Giày thể thao', 'Nữ'),
           ('GT02', 'Giày tây', 'Nữ'),
           ('GSD02', 'Giày sandal', 'Nữ'),
           ('ASM03','Áo sơ mi', 'Unisex'),
           ('AT03', 'Áo thun', 'Unisex'),
           ('AK03', 'Áo khoác', 'Unisex'),
           ('QJ03', 'Quần jean', 'Unisex'),
           ('QS03', 'Quần short', 'Unisex'),
           ('QD03', 'Quần đùi', 'Unisex')
"""
execute_query(connection, insert_query)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Đã đóng kết nối MySQL


### Draft

In [ ]:
# Dữ liệu cho Staff
positions = ['Nhân viên bán hàng', 'Quản lý kho', 'Thu ngân']
staffs = [(f"NV{i+1:02d}", fake.name(), random.choice(positions), random.uniform(5000000, 20000000)) for i in range(num_staff)]

staff_query = """
INSERT INTO Staff (StaffID, Name, Position, Salary) VALUES (%s, %s, %s, %s)
"""

# Dữ liệu cho Orders
orders = [(f"DH{i+1:02d}", random.choice(customers)[0], random.choice(staffs)[0],
           fake.date_time_this_year(), random.uniform(200000, 2000000), None, random.choice(['Cash', 'Card', 'Momo'])) for i in range(num_orders)]

order_query = """
INSERT INTO Orders (OrderID, CustomerID, StaffID, DateTime, TotalPrice, DiscountID, Payment_method) VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Shipment
shipments = [(f"GH{i+1:02d}", random.choice(orders)[0], random.choice(customers)[0],
              random.choice(['Đang giao hàng', 'Đã giao', 'Chờ xử lý']), fake.company(), fake.sentence()) for i in range(num_shipments)]

shipment_query = """
INSERT INTO Shipment (ShipID, OrderID, CustomerID, State, Shipper, Shipper_info) VALUES (%s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Discount
discounts = [(f"DSC{i+1:02d}", fake.word(), random.randint(5, 50), fake.date_time_this_year(), fake.date_time_this_year()) for i in range(num_discounts)]

discount_query = """
INSERT INTO Discount (DiscountID, Name, Percentage, Start_time, End_time) VALUES (%s, %s, %s, %s, %s)
"""

# Dữ liệu cho OrderDetail (tránh trùng khóa chính)
order_detail_set = set()
order_details = []
while len(order_details) < num_order_details:
    order_id = random.choice(orders)[0]
    product_id = random.choice(products)[0]
    if (order_id, product_id) not in order_detail_set:
        order_detail_set.add((order_id, product_id))
        order_details.append((order_id, product_id, random.randint(1, 5)))

order_detail_query = """
INSERT INTO OrderDetail (OrderID, ProductID, Quantity) VALUES (%s, %s, %s)
"""
